# Phase 3 RQ1 v2 — combined analysis

This analysis stage runs on CPU; no model training or GPU is needed. It analyzes the same ten-language Phase 4 roster: `en`, `de`, `es`, `fr`, `sw`, `ar`, `ur`, `bg`, `ru`, and `hi`. Turn Internet on to clone the repository. Attach the private input dataset containing `phase3_rq1_v2_inputs.zip`. For each seed, attach either its complete seed ZIP from an earlier full-seed run or all five order ZIPs (`phase3_rq1_v2_b0_seed_<seed>_order-<n>.zip`). You may mix these formats across seeds, but every seed must have all five orders represented exactly once.

The workflow checks that the seeds, five orders per seed, 15 result records, and data-manifest hashes match the frozen protocol before calculating pooled and per-order correlations, 10,000 language-label permutations, Holm correction, and a random-score control. It exports `phase3_rq1_v2_analysis.zip` under `/kaggle/working`. Review the per-seed metric summaries in the provenance and the per-order correlation table alongside the pooled corrected results; this pilot does not automatically select a prior.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT))
inputs = list(Path("/kaggle/input").rglob("phase3_rq1_v2_inputs.zip"))
result_zips = sorted(Path("/kaggle/input").rglob("phase3_rq1_v2_b0_seed_*.zip"))
if len(inputs) != 1:
    raise FileNotFoundError("Attach exactly one Phase 3 v2 input ZIP; found: " + str(inputs))
if not 3 <= len(result_zips) <= 15:
    raise FileNotFoundError("Attach one complete ZIP per seed or five per-order ZIPs per seed; found: " + str(result_zips))

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_analysis")
command = [
    sys.executable, "-m", "phase3.rq1_v2_workflow", "analyze",
    "--inputs-zip", str(inputs[0]),
    "--output-dir", str(OUTPUT_DIR),
    "--random-repetitions", "10000",
]
for result_zip in result_zips:
    command.extend(["--result-zip", str(result_zip)])
subprocess.run(command, cwd=REPO_ROOT, check=True)

ANALYSIS_ZIP = OUTPUT_DIR / "phase3_rq1_v2_analysis.zip"
if not ANALYSIS_ZIP.is_file():
    raise FileNotFoundError(f"Analysis bundle was not created: {ANALYSIS_ZIP}")
print(f"Download the combined analysis from Kaggle output files: {ANALYSIS_ZIP}")